In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
pd.set_option("display.max_columns",100)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
df1=pd.read_csv("/kaggle/input/titanic/train.csv")
df2=pd.read_csv("/kaggle/input/titanic/test.csv")

In [ ]:
df=pd.concat([df1,df2],ignore_index=True)

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

# EDA

In [ ]:
df.Name.value_counts()

In [ ]:
df["Title"]=df["Name"].str.extract("([A-Za-z]+)\.",expand=False)
df['Title']=df['Title'].replace(['Ms','Mlle'],'Miss')
df['Title']=df['Title'].replace(['Mme','Countess','Lady','Dona'],'Mrs')
df['Title']=df['Title'].replace(['Dr','Major','Col','Capt','Sir','Rev','Jonkheer','Don'],'Mr')

In [ ]:
df['Title'].value_counts()
sns.countplot(df["Title"])

In [ ]:
df["Age"].fillna(df.groupby("Title")["Age"].transform("median"),inplace=True)

In [ ]:
del df["Cabin"]

In [ ]:
df["Fare"].fillna(df["Fare"].median(),inplace=True)

In [ ]:
df['Family']=df['SibSp']+df['Parch']+1

In [ ]:
sns.countplot(df["Embarked"],hue=df["Survived"])

In [ ]:
df["Embarked"]=df["Embarked"].fillna("S")

In [ ]:
df.drop(['Ticket'],axis=1,inplace=True)

In [ ]:
df.drop("Name",axis=1,inplace=True)

In [ ]:
df=pd.get_dummies(df,drop_first=True)

In [ ]:
df.shape

# Prediction 

In [ ]:
def result_func(model,count):
    predict_x=model.predict(df[891:].drop("Survived",axis=1))
    result_dataset=pd.DataFrame()
    result_dataset["PassengerId"]=df[891:]["PassengerId"]
    result_dataset["Predict"]=predict_x
    result_dataset["Survived"]=result_dataset["Predict"].map(lambda s:1 if s>=0.5 else 0 )
    print(result_dataset["Survived"].value_counts().plot.bar())
    result_dataset.drop("Predict",axis=1).to_csv("titanic_deep_learning_result_model{}.csv".format(count),index=False)
    return result_dataset 

In [ ]:
model=Sequential()
model.add(Dense(13,activation='relu'))
model.add(Dense(9,activation='relu'))
model.add(Dense(6,activation='relu'))
model.add(Dense(3,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(loss='binary_crossentropy',optimizer="adam",metrics=["accuracy"])

In [ ]:
model.fit(x,y,epochs=200,batch_size=10,verbose=1)

In [ ]:
model.summary()

In [ ]:
scores=model.evaluate(x,y)

In [ ]:
print("%s: %.2f%%" % (model.metrics_names[1],scores[1]*100))

In [ ]:
result_func(model,1)

In [ ]:
history=model.fit(x,y,epochs=180,batch_size=10,verbose=1)

In [ ]:
fig1=plt.figure(1)
plt.plot(history.history["accuracy"])
plt.plot(history.history["loss"])
plt.title("Model Accuracy")
plt.xlabel("Accuracy")
plt.ylabel("Epoch")
plt.legend(["training", "testing"], loc= "upper left")

In [ ]:
model=Sequential()
model.add(Dense(14,activation='relu'))
model.add(Dense(9,activation='relu'))
model.add(Dense(4,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(loss='binary_crossentropy',optimizer="adam",metrics=["accuracy"])

In [ ]:
model.fit(x,y,validation_split=0.2,epochs=150,batch_size=10,verbose=2)

In [ ]:
scores=model.evaluate(x,y)

In [ ]:
print("%s: %.2f%%" % (model.metrics_names[1],scores[1]*100))

In [ ]:
result_func(model,2)

In [ ]:
history=model.fit(x,y,validation_split=0.2,epochs=200,batch_size=10,verbose=2)

In [ ]:
fig1=plt.figure(1)
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Model Accuracy")
plt.xlabel("Accuracy")
plt.ylabel("Epoch")
plt.legend(["training", "testing"], loc= "upper left")

In [ ]:
fig1=plt.figure(2)
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Model Loss")
plt.xlabel("Loss")
plt.ylabel("Epoch")
plt.legend(["training", "testing"], loc= "upper left")

plt.show